# Explainable AI-Based Air Pollution Prediction and Source Attribution

**Repository:** https://github.com/BhanuDanda-09/Air-Pollution-XAI.git  
**Phases Covered:**
- **Phase 1:** Dataset Inspection, Leak-Free Preprocessing, Exploratory Data Analysis, and Baseline Random Forest Regressor
- **Phase 2:** XGBoost Regressor Development, 5-Fold Cross-Validation Tuning, and Cross-Model Comparison
- **Phase 3:** Explainable AI (XAI) using SHAP (TreeExplainer) and LIME (LimeTabularExplainer)

---
### Notebook Objectives:
1. **Dataset Inspection:** Audit raw dataset `Delhi Final(1).xlsx` (dimensions, data types, missingness, duplicates, physical domains).
2. **Data Preprocessing:** Remove empty sheet artifacts, eliminate 156 duplicate rows to prevent train-test data leakage, build a median imputer pipeline, and split into 80% train / 20% test partitions (`random_state=42`).
3. **Exploratory Data Analysis (EDA):** Generate publication-grade diagnostic visualizations.
4. **Baseline Modeling (Random Forest):** Train a `RandomForestRegressor` pipeline and evaluate on held-out test data.
5. **Phase 2 Modeling (XGBoost):** Optimize an `XGBRegressor` using 5-fold cross-validation (`RandomizedSearchCV`) on training data.
6. **Cross-Model Evaluation:** Compare Random Forest and XGBoost head-to-head on the exact same 406 test samples.
7. **Phase 3 SHAP Explainability:** Calculate SHAP values, beeswarm plots, global importance, and waterfall breakdowns.
8. **Phase 3 LIME Explainability:** Fit local linear surrogate models on representative test instances.
9. **Inference Verification:** Validate prediction on a live unseen sample using both models.


In [ ]:
import os
import sys
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from lime import lime_tabular

# Add project root to sys.path
sys.path.insert(0, os.path.abspath('..'))

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set plot styling
sns.set_theme(style='whitegrid', font='sans-serif')
plt.rcParams['figure.dpi'] = 150
print('Libraries imported successfully.')


## 1. Dataset Loading & Inspection
We load the raw Excel dataset `Delhi Final(1).xlsx` and inspect its properties.


In [ ]:
dataset_path = '../dataset/Delhi Final(1).xlsx'
raw_df = pd.read_excel(dataset_path)

print('Raw Dataset Dimensions:', raw_df.shape)
print('\nAll Detected Columns:', raw_df.columns.tolist())
raw_df.head()


In [ ]:
# Inspect the 6 core variables
core_cols = ['PM2.5', 'RH', 'WD', 'SR', 'BP', 'AT']
core_df = raw_df[core_cols]

print('--- Missing Values in Core Variables ---')
print(core_df.isnull().sum())

print('\n--- Statistical Summary ---')
display(core_df.describe().T)

print('\nExact duplicate rows in core features:', core_df.duplicated().sum())


## 2. Preprocessing & Leakage-Free Train-Test Split

1. **Artifact Removal:** Dropped empty artifact columns `Unnamed: 6` to `Unnamed: 12`.
2. **Deduplication:** Dropped 156 duplicate rows to ensure cross-fold independence and eliminate train-test leakage.
3. **Train-Test Split:** 80% Training (1,621 samples) and 20% Testing (406 samples) with `random_state=42`.
4. **Imputation:** `SimpleImputer(strategy='median')` fitted exclusively on `X_train`.


In [ ]:
# Deduplicate
cleaned_df = core_df.drop_duplicates().reset_index(drop=True)
print('Cleaned Dataset Shape:', cleaned_df.shape)

# Separate Features (X) and Target (y)
features = ['RH', 'WD', 'SR', 'BP', 'AT']
target = 'PM2.5'

X = cleaned_df[features]
y = cleaned_df[target]

# Train-test split (80% train, 20% test, seed=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, shuffle=True
)

print(f'Training Set: {X_train.shape[0]} samples')
print(f'Testing Set:  {X_test.shape[0]} samples')


## 3. Exploratory Data Analysis (EDA) Visualizations


In [ ]:
# Target distribution and boxplot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(cleaned_df['PM2.5'], kde=True, color='#d95f02', bins=35, ax=axes[0], edgecolor='k')
pm_mean = cleaned_df['PM2.5'].mean()
pm_median = cleaned_df['PM2.5'].median()
axes[0].axvline(pm_mean, color='red', linestyle='--', label=f'Mean: {pm_mean:.1f}')
axes[0].axvline(pm_median, color='blue', linestyle='-', label=f'Median: {pm_median:.1f}')
axes[0].set_title('PM2.5 Concentration Distribution', fontweight='bold')
axes[0].set_xlabel('PM2.5 (µg/m³)')
axes[0].legend()

sns.boxplot(x=cleaned_df['PM2.5'], color='#fc8d62', ax=axes[1])
axes[1].set_title('PM2.5 Concentration Boxplot', fontweight='bold')
axes[1].set_xlabel('PM2.5 (µg/m³)')

plt.tight_layout()
plt.show()


In [ ]:
# Pearson correlation heatmap
plt.figure(figsize=(8, 6))
corr = cleaned_df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, center=0, square=True, linewidths=1.0)
plt.title('Pearson Correlation Heatmap', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()


## 4. Phase 1: Baseline Random Forest Regressor


In [ ]:
rf_pipeline = joblib.load('../models/random_forest.pkl')

y_pred_rf = rf_pipeline.predict(X_test)
rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
rf_r2 = r2_score(y_test, y_pred_rf)

print(f'Random Forest Test MAE:  {rf_mae:.2f} µg/m³')
print(f'Random Forest Test RMSE: {rf_rmse:.2f} µg/m³')
print(f'Random Forest Test R²:   {rf_r2:.4f} ({rf_r2*100:.2f}%)')


## 5. Phase 2: XGBoost Regressor


In [ ]:
xgb_pipeline = joblib.load('../models/xgboost_model.pkl')

y_pred_xgb = xgb_pipeline.predict(X_test)
xgb_mae = mean_absolute_error(y_test, y_pred_xgb)
xgb_rmse = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
xgb_r2 = r2_score(y_test, y_pred_xgb)

print(f'XGBoost Test MAE:  {xgb_mae:.2f} µg/m³')
print(f'XGBoost Test RMSE: {xgb_rmse:.2f} µg/m³')
print(f'XGBoost Test R²:   {xgb_r2:.4f} ({xgb_r2*100:.2f}%)')


## 6. Head-to-Head Model Comparison (Random Forest vs. XGBoost)


In [ ]:
comp_table = pd.read_csv('../results/metrics/model_comparison.csv')
display(comp_table)


## 7. Phase 3: SHAP (SHapley Additive exPlanations)

We use `shap.TreeExplainer` on the imputed test observations.


In [ ]:
# Preprocess X_test through pipeline imputer to maintain feature names
imputer = xgb_pipeline.named_steps['imputer']
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=features, index=X_test.index)

# TreeExplainer for XGBoost
xgb_model = xgb_pipeline.named_steps['regressor']
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer(X_test_imp)

print('SHAP values computed successfully. Shape:', shap_values.values.shape)
print('Base Expected Value:', explainer.expected_value)


In [ ]:
# SHAP Beeswarm Summary Plot
plt.figure(figsize=(9, 6))
shap.plots.beeswarm(shap_values, show=True)


In [ ]:
# SHAP Global Feature Importance Bar Plot
plt.figure(figsize=(8, 5))
shap.plots.bar(shap_values, show=True)


In [ ]:
# SHAP Waterfall Plot for Sample #0
shap.plots.waterfall(shap_values[0], show=True)


## 8. Phase 3: LIME (Local Interpretable Model-agnostic Explanations)

We fit a `LimeTabularExplainer` on `X_train` and explain representative test instances.


In [ ]:
def predict_fn(x):
    return xgb_pipeline.predict(pd.DataFrame(x, columns=features))

lime_explainer = lime_tabular.LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=features,
    mode='regression',
    discretize_continuous=False,
    random_state=42
)

# Explain test sample #0
exp = lime_explainer.explain_instance(X_test.iloc[0].values, predict_fn, num_features=5)
fig = exp.as_pyplot_figure()
plt.title(f'XGBoost LIME Explanation (Sample #0)', fontweight='bold')
plt.show()


## 9. Cross-Method Comparison (SHAP vs. LIME)


In [ ]:
exp_comp = pd.read_csv('../results/metrics/explainability_comparison.csv')
display(exp_comp)


## 10. Crucial Scientific Limitations

1. **Model Reliance vs. Physical Causation:** SHAP and LIME explain **how the model makes predictions**, not physical causation.
2. **No Chemical Source Apportionment:** Feature importance cannot distinguish physical emission sources (e.g., vehicle exhaust vs. crop burning vs. power plants) without chemical tracer speciation.
3. **No Timestamp Sequences:** The dataset does not contain temporal headers and is modeled as cross-sectional tabular data.
